# 01 · Carga y limpieza de datos

**Proyecto semestral IMT2200, Grupo 5:** *¿Mide el NEM lo mismo para todos?*

Lee las 16 tablas del MINEDUC (4 tipos × procesos 2023-2026), aplica las
decisiones de limpieza y guarda el panel en `data/processed/panel_completo.parquet`,
que es la entrada de `02_eda`. La Decisión 5 (egreso reciente) se aplica en
`02_eda`.

In [1]:
# Librerías, IDs de los 16 CSV en Google Drive y una función de limpieza.
import os

import numpy as np
import pandas as pd

ANIOS = [2026, 2025, 2024, 2023]
URL = ('https://drive.usercontent.google.com/download?id={}'
       '&export=download&authuser=0&confirm=t')
IDS_DRIVE = {
    'IP-2026': '1RurrECku8Ez_Rl3RlOfjP1SopxgpR6RC',
    'IP-2025': '1gAmWo_ujbhjJPr_vmWOhj-oXjycCk535',
    'IP-2024': '1gAx17O6USHw0AlXOsFSz9SFSG44TuUPY',
    'IP-2023': '1PcOS0fPcAMpwlunRqz9vJRHBwz1lTsLG',
    'SD-2026': '1qAV7_q6Z07lsJGgRxlb0W92vSeZmSm06',
    'SD-2025': '1b3QBKylAHGgw0CcXJHY3DfacDTHYUEAU',
    'SD-2024': '1xx6zZNFkAwUC4qwhVe8zpB6qHie8M6Se',
    'SD-2023': '15quWF_o5R4OO2caH9rF6hmy8dbExnXVH',
    'PS-2026': '1F7cv0-gQLHbl2n7TvywUt9KedzVrr8fD',
    'PS-2025': '1DN-boiGtFXiL8_R9-c2j642ftp0fu25x',
    'PS-2024': '1TDi0q0l12_hGQTIC_f0X8zXGokq68tO1',
    'PS-2023': '188bcegEIs_quVaWDSQ0rc6zV4Q3kBJ96',
    'M-2026': '1oUQx9Sd-VFLRmhKoM_gFQMARY1AJS4yx',
    'M-2025': '1uuj7y793AXdgiFhy5GFpqkAF_tqVgEWH',
    'M-2024': '1aCClSoUgHLtBOIKbzpOXi29zj2L8cieS',
    'M-2023': '1IRDbdWaqWf-_mO7usfwdmkQ74iwuUmoi',
}


def a_numero(serie: pd.Series) -> pd.Series:
    # Lo que no es número (p. ej., una celda con ' ') queda como NaN. Si la
    # columna quedó como texto, los decimales vienen con coma.
    if pd.api.types.is_numeric_dtype(serie):
        return serie.astype(float)
    texto = serie.astype(str).str.replace(',', '.')
    return pd.to_numeric(texto, errors='coerce').astype(float)

## Carga de los datos

Los 16 CSV se leen directo desde Google Drive, sin guardar copias locales, así
que esta celda tarda algunos minutos. Usan `;` como separador y coma decimal; los
nombres de columnas se limpian para que sean iguales en todos los años.

In [2]:
# Lee los 16 CSV y cuenta las filas de cada tabla (fila) por año (columna).
tablas = {'IP': {}, 'SD': {}, 'PS': {}, 'M': {}}
filas = {}
for sigla in tablas:
    filas[sigla] = {}
    for anio in ANIOS:
        df = pd.read_csv(URL.format(IDS_DRIVE[f'{sigla}-{anio}']), sep=';',
                         decimal=',', low_memory=False)
        df.columns = (df.columns.str.replace('﻿', '').str.strip()
                      .str.upper())
        tablas[sigla][anio] = df
        filas[sigla][anio] = len(df)
display(pd.DataFrame(filas).T)
ip, sd, ps, m = tablas['IP'], tablas['SD'], tablas['PS'], tablas['M']

,2026,2025,2024,2023
IP,320087,313759,309350,296812
SD,320087,313759,309350,296812
PS,1694081,1738995,1733891,187226
M,146873,140687,134492,128895


## Decisión 1: unicidad de `MRUN` y columnas entre años

- `IP`, `SD` y `M` tienen una fila por estudiante; `PS` tiene una por preferencia
  (hasta 20), así que se resume antes de unir (Decisión 2).
- Las diferencias de columnas no afectan al análisis: **PS-2023** viene en formato
  ancho (se pasa a largo en la Decisión 2), IP-2023 y 2024 nombran distinto algunas
  columnas de invierno y del proceso anterior, de `M` solo se usa `MRUN` y SD-2023
  trae una encuesta adicional.

In [3]:
# Decisión 1: filas con MRUN repetido en 2026 y columnas que faltan o sobran
# en cada año respecto de 2026.
for sigla in tablas:
    repetidas = tablas[sigla][2026]['MRUN'].duplicated(keep=False).sum()
    print(f'{sigla}-2026: {repetidas} filas con MRUN repetido')
    columnas_2026 = set(tablas[sigla][2026].columns)
    for anio in [2025, 2024, 2023]:
        columnas = set(tablas[sigla][anio].columns)
        faltan = sorted(columnas_2026 - columnas)
        sobran = sorted(columnas - columnas_2026)
        if faltan or sobran:
            print(f'   {anio}: faltan {len(faltan)} (ej.: {faltan[:3]}), '
                  f'sobran {len(sobran)} (ej.: {sobran[:3]})')

IP-2026: 0 filas con MRUN repetido
   2024: faltan 2 (ej.: ['MATE1_INV_ANTERIOR', 'MATE2_INV_ANTERIOR']), sobran 1 (ej.: ['MATE_INV_ANTERIOR'])
   2023: faltan 21 (ej.: ['CIEN_INV_ANTERIOR', 'CLEC_INV_ANTERIOR', 'CORRECTAS_INV_M1']), sobran 7 (ej.: ['CORRECTAS_INV_MATE', 'ERRADAS_INV_MATE', 'FORMA_INV_MATE'])
SD-2026: 0 filas con MRUN repetido
   2023: faltan 0 (ej.: []), sobran 29 (ej.: ['CONEXION_INSUFICIENTE', 'DISPOSITIVO_1', 'DISPOSITIVO_2'])
PS-2026: 1685805 filas con MRUN repetido
   2023: faltan 7 (ej.: ['COD_CARRERA_PREF', 'ESTADO_PREF', 'LUGAR_PREF']), sobran 82 (ej.: ['COD_CARRERA_PREF_01', 'COD_CARRERA_PREF_02', 'COD_CARRERA_PREF_03'])
M-2026: 0 filas con MRUN repetido
   2023: faltan 4 (ej.: ['LUGAR_EN_LA_LISTA', 'PREFERENCIA', 'PUNTAJE_PONDERADO']), sobran 3 (ej.: ['FECHA_MAT', 'NOMBRE_INSTITUCION', 'VIA_ADMISION'])


## Decisión 2: postulaciones a una fila por estudiante

Códigos de `ESTADO_PREF` según el libro de códigos de Postulación 2026 (`docs/`,
Anexo I): `24` = en lista de seleccionados, `25` = en lista de espera, `26` =
seleccionado en una preferencia anterior.

- **Postulación válida:** alguna preferencia en 24, 25 o 26.
- **Seleccionado:** alguna preferencia en 24 o 26 (en PS-2023 a veces falta el 24).
- **Puntaje ponderado máximo:** el mayor `PTJE_PREF` entre sus preferencias.

**Control:** PS-2023 trae `SITUACION_POSTULANTE` (`P` = al menos una postulación en
selección o espera), que debería coincidir con la postulación válida.

In [4]:
# PS-2023 viene en formato ancho (ESTADO_PREF_01 a ESTADO_PREF_20, y lo
# mismo para PTJE_PREF): se apilan las 20 preferencias en formato largo.
partes = []
for n in range(1, 21):
    parte = ps[2023][['MRUN', f'ESTADO_PREF_{n:02d}', f'PTJE_PREF_{n:02d}']]
    parte.columns = ['MRUN', 'ESTADO_PREF', 'PTJE_PREF']
    partes.append(parte)
ps_2023 = pd.concat(partes, ignore_index=True)
# Las preferencias vacías vienen con ESTADO_PREF = 0 y se eliminan.
estado = pd.to_numeric(ps_2023['ESTADO_PREF'], errors='coerce')
ps_largo = {2026: ps[2026], 2025: ps[2025], 2024: ps[2024],
            2023: ps_2023[estado.notna() & (estado != 0)]}

# Una fila por estudiante: el máximo de True/False es "alguna cumple".
postulacion = {}
for anio in ANIOS:
    estado = pd.to_numeric(ps_largo[anio]['ESTADO_PREF'], errors='coerce')
    resumen = pd.DataFrame({
        'MRUN': ps_largo[anio]['MRUN'],
        'postulacion_valida': estado.isin([24, 25, 26]),
        'seleccionado': estado.isin([24, 26]),
        'ptje_ponderado_max': a_numero(ps_largo[anio]['PTJE_PREF']),
    })
    postulacion[anio] = resumen.groupby('MRUN', as_index=False).max()
    print(f"PS {anio}: {len(postulacion[anio])} postulantes, "
          f"{postulacion[anio]['seleccionado'].sum()} seleccionados")

PS 2026: 189399 postulantes, 161907 seleccionados
PS 2025: 189982 postulantes, 158218 seleccionados


PS 2024: 188707 postulantes, 150492 seleccionados
PS 2023: 187226 postulantes, 142902 seleccionados


In [5]:
# Control (solo 2023): postulación válida vs SITUACION_POSTULANTE = 'P'.
control = ps[2023][['MRUN', 'SITUACION_POSTULANTE']].merge(
    postulacion[2023], on='MRUN', how='left')
es_p = control['SITUACION_POSTULANTE'].astype(str).str.strip() == 'P'
coinciden = (es_p == control['postulacion_valida']).mean()
print(f'Coinciden en el {coinciden:.2%} de los {len(control)} postulantes')

Coinciden en el 100.00% de los 187226 postulantes

## Decisiones 3 y 4: panel por año, tipos y no respuesta de ingreso

- **Panel:** `IP` es el universo de inscritos, así que los `merge` son `left` para no
  perder a quienes no postularon o no se matricularon.
- **Tipos:** varias columnas vienen como texto por celdas en blanco; sin corregirlo,
  filtros como `ANYO_DE_EGRESO == 2025` fallan en silencio. Lo que no es número pasa
  a `NaN`.
- **Ingreso:** el código 99 es "no responde"; pasa a `NaN` sin eliminar filas (en el
  EDA es una categoría propia).
- **Sexo:** se usa `SEXO` de la base Socioeconómica; se verificó que coincide en un
  100% con `COD_SEXO` de la base de Inscritos.

In [6]:
# Columnas que se pasan a número: las de puntajes (4 pruebas, regular e
# invierno, proceso actual y anterior) y otras 8.
PUNTAJES = []
for prueba in ['CLEC', 'MATE1', 'HCSOC', 'CIEN']:
    for proceso in ['ACTUAL', 'ANTERIOR']:
        PUNTAJES += [f'{prueba}_REG_{proceso}', f'{prueba}_INV_{proceso}']
NUMERICAS = ['RBD', 'CODIGO_ENS', 'DEPENDENCIA', 'SEXO', 'ANYO_DE_EGRESO',
             'PROMEDIO_NOTAS', 'PTJE_NEM',
             'INGRESO_PERCAPITA_GRUPO_FA'] + PUNTAJES

paneles = []
perdidos = {}
for anio in ANIOS:
    panel = ip[anio].merge(sd[anio], on='MRUN', how='left',
                           suffixes=('', '_sd'))
    panel = panel.merge(postulacion[anio], on='MRUN', how='left')
    panel = panel.copy()  # evita un aviso de rendimiento de pandas
    panel['postulo'] = panel['MRUN'].isin(postulacion[anio]['MRUN'])
    # Quienes no postularon quedan como NaN tras el merge: pasan a False.
    for col in ['postulacion_valida', 'seleccionado']:
        panel[col] = panel[col].fillna(False).astype(bool)
    panel['matriculado'] = panel['MRUN'].isin(m[anio]['MRUN'])
    panel['ANIO'] = anio

    perdidos[anio] = {}
    for col in NUMERICAS:
        # Algunas columnas de invierno o del proceso anterior no existen en
        # 2023 y 2024.
        if col in panel.columns:
            numero = a_numero(panel[col])
            perdidos[anio][col] = (numero.isna() & panel[col].notna()).sum()
            panel[col] = numero
    ingreso = panel['INGRESO_PERCAPITA_GRUPO_FA']
    panel['INGRESO_PERCAPITA_GRUPO_FA'] = ingreso.replace(99, np.nan)
    paneles.append(panel)

print('Celdas no numéricas (en blanco) que pasaron a NaN, por columna y año:')
display(pd.DataFrame(perdidos))
sin_ingreso = paneles[0]['INGRESO_PERCAPITA_GRUPO_FA'].isna().mean()
print(f'2026: {sin_ingreso:.1%} no declara ingreso (código 99)')

Celdas no numéricas (en blanco) que pasaron a NaN, por columna y año:


,2026,2025,2024,2023
RBD,3200,5102,4932.0,2658.0
CODIGO_ENS,3200,5102,4932.0,2658.0
DEPENDENCIA,3200,5102,4932.0,2658.0
SEXO,0,0,0.0,0.0
ANYO_DE_EGRESO,218,137,189.0,334.0
PROMEDIO_NOTAS,0,0,0.0,0.0
PTJE_NEM,0,0,0.0,0.0
INGRESO_PERCAPITA_GRUPO_FA,0,0,0.0,0.0
CLEC_REG_ACTUAL,19496,19664,21753.0,21598.0
CLEC_INV_ACTUAL,288070,282692,279286.0,263432.0


2026: 27.6% no declara ingreso (código 99)


## Decisión 6: concatenación y guardado

Se apilan los cuatro años y se guardan solo las columnas que usa el EDA, en formato
Parquet (más liviano que CSV y conserva los tipos de datos).

In [7]:
# Decisión 6: apila los 4 años y guarda el panel para 02_eda.
COLUMNAS_PANEL = [
    'MRUN', 'ANIO', 'RBD', 'CODIGO_ENS', 'DEPENDENCIA', 'RAMA_EDUCACIONAL',
    'SEXO', 'ANYO_DE_EGRESO', 'PROMEDIO_NOTAS', 'PTJE_NEM',
    'INGRESO_PERCAPITA_GRUPO_FA', 'postulo', 'postulacion_valida',
    'seleccionado', 'ptje_ponderado_max', 'matriculado',
] + PUNTAJES
panel_completo = pd.concat(paneles, ignore_index=True)[COLUMNAS_PANEL]
os.makedirs('../data/processed', exist_ok=True)
panel_completo.to_parquet('../data/processed/panel_completo.parquet',
                          index=False)
print(f'Guardado: {len(panel_completo)} inscritos, '
      f'{len(COLUMNAS_PANEL)} columnas')

Guardado: 1240008 inscritos, 32 columnas
